# Notebook 29 — Operator Composition Dynamics

This notebook extends the residual manifold operator sequence by studying **operator composition**.

Notebook 28 studied perturbation, phase boundaries, and spectral persistence. Notebook 29 studies how residual operators interact when composed in sequence:

\[
\mathcal{O}_{\mathrm{chain}}
= R_{f_1} R_{f_2} \cdots R_{f_m},
\qquad
R_f = \exp(-\alpha L_f).
\]

The goal is to measure:

- composition-order asymmetry,
- stability accumulation across chain depth,
- spectral transport resonance,
- phase-lock continuity between chained operator states,
- residual spectral flow under repeated operator application.

Outputs are written to `figures/`, `results/`, and `exports/`, with an optional Colab download zip at the end.


In [ ]:
# Notebook 29 setup
from pathlib import Path
import json, zipfile, math, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from scipy.linalg import expm
from scipy.sparse.csgraph import laplacian as csgraph_laplacian
from sklearn.decomposition import PCA

BASE_DIR = Path('/content') if Path('/content').exists() else Path('.')
FIGURES_DIR = BASE_DIR / 'figures'
RESULTS_DIR = BASE_DIR / 'results'
EXPORTS_DIR = BASE_DIR / 'exports'
for d in [FIGURES_DIR, RESULTS_DIR, EXPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(9423)
plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

print('Directories ready:')
print('FIGURES_DIR =', FIGURES_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
print('EXPORTS_DIR =', EXPORTS_DIR)


## 1. Topology-family operators

We construct five topology-family graph operators on a shared node count:

- ring lattice,
- small world,
- Erdős–Rényi,
- scale free,
- modular clustered.

For each graph family we compute:

\[
L_f = I - D^{-1/2} A_f D^{-1/2}
\]

and its diffusion operator:

\[
R_f(\alpha)=\exp(-\alpha L_f).
\]


In [ ]:
def largest_component_graph(G):
    if nx.is_connected(G):
        return G.copy()
    comp = max(nx.connected_components(G), key=len)
    H = G.subgraph(comp).copy()
    return nx.convert_node_labels_to_integers(H)


def make_graph_family(name, n=72, seed=0):
    rng = np.random.default_rng(seed)
    if name == 'ring lattice':
        G = nx.watts_strogatz_graph(n=n, k=6, p=0.0, seed=seed)
    elif name == 'small world':
        G = nx.watts_strogatz_graph(n=n, k=6, p=0.15, seed=seed)
    elif name == 'Erdős–Rényi':
        p = 6 / (n - 1)
        G = nx.erdos_renyi_graph(n=n, p=p, seed=seed)
        G = largest_component_graph(G)
        # pad if needed by retrying
        tries = 0
        while G.number_of_nodes() < n and tries < 20:
            seed += 11
            G = nx.erdos_renyi_graph(n=n, p=p, seed=seed)
            G = largest_component_graph(G)
            tries += 1
    elif name == 'scale free':
        G = nx.barabasi_albert_graph(n=n, m=3, seed=seed)
    elif name == 'modular clustered':
        sizes = [n//4, n//4, n//4, n - 3*(n//4)]
        p_in, p_out = 0.22, 0.025
        probs = [[p_in if i == j else p_out for j in range(4)] for i in range(4)]
        G = nx.stochastic_block_model(sizes, probs, seed=seed)
        G = largest_component_graph(G)
    else:
        raise ValueError(name)
    return nx.convert_node_labels_to_integers(G)


def normalized_laplacian_dense(G, n_target=None):
    A = nx.to_numpy_array(G, dtype=float)
    # If a rare ER/SBM component is smaller, embed into target dimension with isolated small diagonal regularization.
    if n_target is not None and A.shape[0] < n_target:
        B = np.zeros((n_target, n_target), dtype=float)
        B[:A.shape[0], :A.shape[1]] = A
        A = B
    deg = A.sum(axis=1)
    deg_safe = np.where(deg > 0, deg, 1.0)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(deg_safe))
    L = np.eye(A.shape[0]) - D_inv_sqrt @ A @ D_inv_sqrt
    # isolated nodes get zeroed diagonal contribution after safe degree; restore normalized-lap identity convention lightly
    return (L + L.T) / 2

families = ['ring lattice', 'small world', 'Erdős–Rényi', 'scale free', 'modular clustered']
short = {'ring lattice':'ring', 'small world':'small', 'Erdős–Rényi':'ER', 'scale free':'scale', 'modular clustered':'modular'}
N = 72
alpha = 0.65

operators = {}
for i, fam in enumerate(families):
    G = make_graph_family(fam, n=N, seed=2026 + 17*i)
    L = normalized_laplacian_dense(G, n_target=N)
    R = expm(-alpha * L)
    operators[fam] = {'G': G, 'L': L, 'R': R, 'eig': np.sort(np.linalg.eigvalsh(L))}

for fam in families:
    G = operators[fam]['G']
    print(f'{fam:18s} nodes={G.number_of_nodes():3d} edges={G.number_of_edges():4d}')


## 2. Spectral metrics for composed operators

For each operator chain we compute a spectrum from the symmetrized composed operator:

\[
C = R_{f_1} R_{f_2}\cdots R_{f_m},
\qquad
C_s = \frac{C+C^T}{2}.
\]

We then extract:

- spectral entropy,
- eigengap mass,
- stability score,
- spectral vector for PCA phase projection.


In [ ]:
def spectral_vector_from_matrix(M, k=24):
    S = (M + M.T) / 2
    vals = np.linalg.eigvalsh(S)
    vals = np.sort(np.real(vals))[::-1]  # diffusion operator: largest modes first
    vals = vals[:k]
    if len(vals) < k:
        vals = np.pad(vals, (0, k-len(vals)))
    return vals


def entropy_from_values(vals):
    x = np.abs(np.asarray(vals, dtype=float))
    x = x / (x.sum() + 1e-12)
    return float(-(x * np.log(x + 1e-12)).sum())


def eigengap_mass(vals):
    vals = np.asarray(vals, dtype=float)
    gaps = np.abs(np.diff(vals))
    if gaps.sum() <= 1e-12:
        return 0.0
    # concentration of dominant gaps: high when structure is partitioned by a few gaps
    p = gaps / gaps.sum()
    return float((p**2).sum())


def stability_score(vals):
    vals = np.asarray(vals, dtype=float)
    vals = np.abs(vals)
    if vals.sum() <= 1e-12:
        return 0.0
    cumulative = np.cumsum(vals) / vals.sum()
    # stable if mass is not entirely fragmented into tail but has structured leading support
    early = cumulative[min(5, len(cumulative)-1)]
    ent = entropy_from_values(vals) / np.log(len(vals) + 1e-12)
    gap = eigengap_mass(vals)
    score = 0.45*early + 0.35*(1-ent) + 0.20*gap
    return float(np.clip(score, 0, 1))


def compose_chain(chain, alpha=alpha):
    C = np.eye(N)
    for fam in chain:
        C = C @ operators[fam]['R']
    return C

# Build repeated-family chains and mixed chains.
chain_records = []
max_depth = 8
for fam in families:
    for depth in range(1, max_depth+1):
        chain = tuple([fam]*depth)
        M = compose_chain(chain)
        vals = spectral_vector_from_matrix(M)
        chain_records.append({
            'chain_type': 'repeated',
            'label': fam,
            'chain': ' -> '.join(short[x] for x in chain),
            'depth': depth,
            'entropy': entropy_from_values(vals),
            'eigengap_mass': eigengap_mass(vals),
            'stability': stability_score(vals),
            **{f'spec_{i}': vals[i] for i in range(len(vals))}
        })

mixed_chains = [
    ('ring lattice','small world'),
    ('small world','ring lattice'),
    ('ring lattice','scale free'),
    ('scale free','ring lattice'),
    ('modular clustered','scale free'),
    ('scale free','modular clustered'),
    ('Erdős–Rényi','small world'),
    ('small world','Erdős–Rényi'),
    ('ring lattice','small world','Erdős–Rényi'),
    ('scale free','modular clustered','Erdős–Rényi'),
]
for chain in mixed_chains:
    for depth in range(1, max_depth+1):
        expanded = tuple(chain * math.ceil(depth/len(chain)))[:depth]
        M = compose_chain(expanded)
        vals = spectral_vector_from_matrix(M)
        chain_records.append({
            'chain_type': 'mixed',
            'label': ' -> '.join(short[x] for x in chain),
            'chain': ' -> '.join(short[x] for x in expanded),
            'depth': depth,
            'entropy': entropy_from_values(vals),
            'eigengap_mass': eigengap_mass(vals),
            'stability': stability_score(vals),
            **{f'spec_{i}': vals[i] for i in range(len(vals))}
        })

metrics_df = pd.DataFrame(chain_records)
metrics_path = RESULTS_DIR / '29_operator_chain_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
metrics_df.head()


## 3. Figure 1 — Operator composition flow trajectories

This figure projects composed-operator spectra into a two-dimensional phase space. Arrows show evolution as composition depth increases.


In [ ]:
spec_cols = [c for c in metrics_df.columns if c.startswith('spec_')]
X = metrics_df[spec_cols].values
pca = PCA(n_components=2, random_state=0)
Z = pca.fit_transform(X)
metrics_df['pc1'] = Z[:,0]
metrics_df['pc2'] = Z[:,1]

fig, ax = plt.subplots(figsize=(11, 7))
plot_df = metrics_df[(metrics_df['chain_type']=='repeated')]
for fam in families:
    sub = plot_df[plot_df['label']==fam].sort_values('depth')
    ax.plot(sub['pc1'], sub['pc2'], marker='o', linewidth=2, label=fam)
    for i in range(len(sub)-1):
        x0, y0 = sub.iloc[i][['pc1','pc2']]
        x1, y1 = sub.iloc[i+1][['pc1','pc2']]
        ax.annotate('', xy=(x1,y1), xytext=(x0,y0), arrowprops=dict(arrowstyle='->', alpha=0.55))
    last = sub.iloc[-1]
    ax.text(last['pc1'], last['pc2'], f"d={int(last['depth'])}", fontsize=9)

ax.axhline(0, linestyle='--', alpha=0.35)
ax.axvline(0, linestyle='--', alpha=0.35)
ax.set_title('Residual operator composition flow trajectories')
ax.set_xlabel('composition phase PC1')
ax.set_ylabel('composition phase PC2')
ax.legend()
fig.tight_layout()
out = FIGURES_DIR / '29_operator_composition_flow.png'
fig.savefig(out, bbox_inches='tight')
plt.show()
print('saved', out)


## 4. Composition ordering asymmetry

Composition order can matter:

\[
R_A R_B \ne R_B R_A.
\]

We measure this through normalized Frobenius asymmetry:

\[
A_{ij} = \frac{\|R_iR_j - R_jR_i\|_F}{\|R_iR_j\|_F + \|R_jR_i\|_F}.
\]


In [ ]:
asym = np.zeros((len(families), len(families)))
for i, fa in enumerate(families):
    for j, fb in enumerate(families):
        A = operators[fa]['R'] @ operators[fb]['R']
        B = operators[fb]['R'] @ operators[fa]['R']
        asym[i,j] = np.linalg.norm(A-B, 'fro') / (np.linalg.norm(A,'fro') + np.linalg.norm(B,'fro') + 1e-12)

asym_df = pd.DataFrame(asym, index=families, columns=families)
asym_path = RESULTS_DIR / '29_composition_asymmetry_matrix.csv'
asym_df.to_csv(asym_path)

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(asym, aspect='auto')
ax.set_xticks(range(len(families)))
ax.set_yticks(range(len(families)))
ax.set_xticklabels(families, rotation=45, ha='right')
ax.set_yticklabels(families)
for i in range(len(families)):
    for j in range(len(families)):
        ax.text(j, i, f'{asym[i,j]:.3f}', ha='center', va='center', fontsize=9)
ax.set_title('Operator composition asymmetry matrix')
fig.colorbar(im, ax=ax, label='normalized order asymmetry')
fig.tight_layout()
out = FIGURES_DIR / '29_composition_asymmetry_matrix.png'
fig.savefig(out, bbox_inches='tight')
plt.show()
print('saved', out)


## 5. Stability accumulation curves

Repeated operator composition can retain continuity or accumulate fragmentation. We track mean residual operator stability across composition depth.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for fam in families:
    sub = metrics_df[(metrics_df['chain_type']=='repeated') & (metrics_df['label']==fam)].sort_values('depth')
    ax.plot(sub['depth'], sub['stability'], marker='o', linewidth=2, label=fam)
ax.set_ylim(0, 1.05)
ax.set_title('Operator stability accumulation under repeated composition')
ax.set_xlabel('composition depth')
ax.set_ylabel('residual operator stability')
ax.legend()
fig.tight_layout()
out = FIGURES_DIR / '29_operator_stability_accumulation.png'
fig.savefig(out, bbox_inches='tight')
plt.show()
print('saved', out)


## 6. Spectral transport resonance landscape

We construct a synthetic composition-depth / coupling-strength landscape.

A resonance score is computed from:

- leading spectral retention,
- eigengap concentration,
- entropy moderation.

This gives a phase-style view of operator reinforcement under chain depth.


In [ ]:
strengths = np.linspace(0.05, 1.0, 18)
depths = np.arange(1, 13)
land = np.zeros((len(depths), len(strengths)))
base_fam = 'small world'
for di, depth in enumerate(depths):
    for si, a in enumerate(strengths):
        R = expm(-a * operators[base_fam]['L'])
        C = np.eye(N)
        for _ in range(depth):
            C = C @ R
        vals = spectral_vector_from_matrix(C)
        land[di, si] = stability_score(vals)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(land, origin='lower', aspect='auto', extent=[strengths.min(), strengths.max(), depths.min(), depths.max()])
ax.set_title('Spectral transport resonance landscape')
ax.set_xlabel('operator coupling strength alpha')
ax.set_ylabel('composition depth')
fig.colorbar(im, ax=ax, label='resonance / stability score')
fig.tight_layout()
out = FIGURES_DIR / '29_transport_resonance_landscape.png'
fig.savefig(out, bbox_inches='tight')
plt.show()
print('saved', out)


## 7. Residual phase-lock continuity

For each repeated composition trajectory, compute cosine alignment between successive spectral states:

\[
\mathrm{lock}_t = \frac{\langle s_t, s_{t+1}\rangle}{\|s_t\|\|s_{t+1}\|}.
\]

High phase-lock means successive operator compositions preserve spectral direction.


In [ ]:
phase_records = []
for fam in families:
    sub = metrics_df[(metrics_df['chain_type']=='repeated') & (metrics_df['label']==fam)].sort_values('depth')
    S = sub[spec_cols].values
    for i in range(len(S)-1):
        a, b = S[i], S[i+1]
        lock = float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-12))
        phase_records.append({'family': fam, 'depth': int(sub.iloc[i]['depth']), 'phase_lock': lock})
phase_df = pd.DataFrame(phase_records)
phase_path = RESULTS_DIR / '29_phase_lock_summary.csv'
phase_df.to_csv(phase_path, index=False)

fig, ax = plt.subplots(figsize=(11,6))
for fam in families:
    sub = phase_df[phase_df['family']==fam]
    ax.plot(sub['depth'], sub['phase_lock'], marker='o', linewidth=2, label=fam)
ax.axhline(1/np.sqrt(2), linestyle='--', alpha=0.45, label='45° threshold')
ax.set_ylim(0, 1.04)
ax.set_title('Residual phase-lock continuity under operator composition')
ax.set_xlabel('composition depth')
ax.set_ylabel('successive spectral cosine alignment')
ax.legend()
fig.tight_layout()
out = FIGURES_DIR / '29_phase_lock_continuity_map.png'
fig.savefig(out, bbox_inches='tight')
plt.show()
print('saved', out)


## 8. Mixed-chain comparison

Mixed chains test transport reinforcement and fragmentation accumulation across operator-family sequences.


In [ ]:
mixed = metrics_df[metrics_df['chain_type']=='mixed'].copy()
pivot = mixed.pivot_table(index='label', columns='depth', values='stability', aggfunc='mean')
pivot_path = RESULTS_DIR / '29_mixed_chain_stability.csv'
pivot.to_csv(pivot_path)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(pivot.values, aspect='auto')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f'{pivot.values[i,j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('Mixed operator-chain stability heatmap')
ax.set_xlabel('composition depth')
ax.set_ylabel('operator chain seed')
fig.colorbar(im, ax=ax, label='stability')
fig.tight_layout()
out = FIGURES_DIR / '29_mixed_chain_stability_heatmap.png'
fig.savefig(out, bbox_inches='tight')
plt.show()
print('saved', out)


## 9. Summary export

Export figures, tables, and a compact JSON manifest. The optional Colab download block is included at the end for consistency with previous notebooks.


In [ ]:
summary = {
    'notebook': '29_operator_composition_dynamics.ipynb',
    'description': 'Residual operator composition dynamics: order asymmetry, stability accumulation, resonance landscapes, and phase-lock continuity.',
    'families': families,
    'node_count': N,
    'alpha': alpha,
    'created_outputs': {
        'figures': sorted([p.name for p in FIGURES_DIR.glob('29_*.png')]),
        'results': sorted([p.name for p in RESULTS_DIR.glob('29_*')]),
    },
    'key_metrics': {
        'max_order_asymmetry': float(np.nanmax(asym)),
        'mean_repeated_chain_stability': float(metrics_df[metrics_df.chain_type=='repeated']['stability'].mean()),
        'mean_phase_lock': float(phase_df['phase_lock'].mean()),
    }
}

summary_path = RESULTS_DIR / '29_operator_composition_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))

md_lines = [
    '# Notebook 29 Results Overview',
    '',
    'Notebook 29 studies residual operator composition dynamics.',
    '',
    '## Figures',
]
for f in summary['created_outputs']['figures']:
    md_lines.append(f'- `figures/{f}`')
md_lines += ['', '## Results']
for f in summary['created_outputs']['results']:
    md_lines.append(f'- `results/{f}`')
md_lines += ['', '## Key metrics', '```json', json.dumps(summary['key_metrics'], indent=2), '```']

overview_path = EXPORTS_DIR / '29_results_overview.md'
overview_path.write_text('\n'.join(md_lines))

print(json.dumps(summary, indent=2))
print('overview:', overview_path)


In [ ]:
# Package Notebook 29 outputs for download.
manifest = {
    'notebook': '29_operator_composition_dynamics.ipynb',
    'created_outputs': {
        'figures': sorted([p.name for p in FIGURES_DIR.glob('29_*.png')]),
        'results': sorted([p.name for p in RESULTS_DIR.glob('29_*')]),
        'exports': ['29_results_overview.md'],
    },
}
manifest_path = EXPORTS_DIR / '29_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))

zip_path = EXPORTS_DIR / '29_operator_composition_dynamics_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in FIGURES_DIR.glob('29_*.png'):
        z.write(p, arcname=f'figures/{p.name}')
    for p in RESULTS_DIR.glob('29_*'):
        z.write(p, arcname=f'results/{p.name}')
    z.write(EXPORTS_DIR / '29_results_overview.md', arcname='exports/29_results_overview.md')
    z.write(manifest_path, arcname='exports/29_manifest.json')

print('Wrote:', zip_path)
print('Zip size MB:', zip_path.stat().st_size / 1e6)

# Optional Colab download
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Colab download skipped. Download manually from:', zip_path)
